In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from ccipy.utils.cci_logger import CCILogger
from ccipy.omero.cci_omero_connection import OmeroConnection
from ccipy.omero.omero_getter_ctx import OmeroGetterCtx
from ccipy.omero.omero_colors import omero_rint_to_rgba
from ccipy.utils.roi_geometry import RoiGeometry
from omero.rtypes import unwrap

2026-10-27 13:50:17,212 DEBUG [              omero.util.TempFileManager] (MainThread) Added file /home/xgupke/omero/tmp/.lock_testpkghg0ea.tmp
2026-10-27 13:50:17,214 DEBUG [              omero.util.TempFileManager] (MainThread) Chose global tmpdir: /home/xgupke/omero/tmp
2026-10-27 13:50:17,216 DEBUG [              omero.util.TempFileManager] (MainThread) Using temp dir: /home/xgupke/omero/tmp/omero_xgupke/3014263


In [18]:
# dataset_id = 1226
# image_ids_tails = [15668, 15643, 15633, 15633, 15664, 15686]
# image_ids_spikes = [15727, 15693, 15653, 15686]
# the above ones have already been downloaded and will be used for training lodestar.

image_ids_tails = [20133, 20112, 20113, 20114, 20109, 20084, 20122, 20074, 21043] + [15668, 15643, 15633, 15633, 15664, 15686] # len 9 
image_ids_spikes = [20111, 20109, 20091, 20994, 15016, 15488, 15695, 15653, 15693, 15727] + [15727, 15693, 15653, 15686] # len 10

image_ids_spikes_new = [20109, 20124, 20051, 20095, 19751, 19792, 19706, 16285, 16291, 16331, 16370, 16354, 16357, 16356, 16302, 16325, 16281, 18549, 18553, 18552, 18864]


dataset_ids = [2251, 2251, 2251, 2251, 2251, 2251, 2251, 2251, 2408, 2251, 2251, 2251, 2408, 1159, 1214, 1226, 1226, 1226, 1226] + [1226] * 10
# these ones will be used for validation

# in omero, both rare types are in purple roi boxes
# it seems that fetching images only requires the image id, not the dataset id
# dataset id is used during train and annotation scripts to find the required image ids

In [19]:
len(dataset_ids), len(image_ids_tails), len(image_ids_spikes)

(29, 15, 14)

In [20]:
from common import init, downscale_data, download_downscaled_image

In [21]:
from pathlib import Path

In [22]:
'''
Let's start simple, open an image ID and print its ROIs
'''

use_test_host = False
group = "Emma-Josefsson-Lab"
session_token = "30875857-f342-4083-ad20-71578168e6df" # paste token here
connection = init(session_token, group, use_test_host=use_test_host)
rare_colour = -1744764929

# create a directory to store the downloaded images
images_dir = Path(f"datafiles/images/rare_granules")
images_dir.mkdir(exist_ok=True, parents=True)
output_file = images_dir / "rare_granules_raw_correct.csv"

save_images = True

with OmeroGetterCtx(connection) as getter:

    csv_file = open(output_file, "a")
    # csv_file.write("dataset_id,")
    # csv_file.write("image name,")
    # csv_file.write("image id,")
    # csv_file.write("pixel size,")
    # csv_file.write("pixel unit,")
    # csv_file.write("class name,")
    # csv_file.write("color,")
    # csv_file.write("x topleft pixels,")
    # csv_file.write("y topleft pixels,")
    # csv_file.write("width pixels,")
    # csv_file.write("height pixels\n")

    already_saved = image_ids_tails + image_ids_spikes
    granule_type = "spike"
    for i,img_id in enumerate(image_ids_spikes_new):

        # if i < len(image_ids_tails):
        #     granule_type = "tail"
        # else:
        #     granule_type = "spike"

        if img_id in already_saved:
            continue

        print('')
        img = getter.conn.get_image(img_id)
        img_name = img.getName()
        if not (img_name.endswith("ome.tif") or img_name.endswith("ome.tiff")):
            CCILogger.info(f"Skipping image {img_id} with name {img_name} as it is not an OME-TIF.")
            continue

        img_name_noext = img_name.split('.')[-3]

        # CCILogger.info(f"Downloading image {img_id} with name {img_name}.")
        img = connection.get_image(img_id)
        img_width = img.getSizeX()
        img_height = img.getSizeY()
        pixels = img.getPrimaryPixels()._obj
        px_x = pixels.getPhysicalSizeX()
        px_y = pixels.getPhysicalSizeY()
        px_unit = px_y.getUnit()
        pixels = img.getPrimaryPixels()

        if save_images:
            img_arr = pixels.getPlane(
                theZ=0,
                theC=0,
                theT=0
            )

            print('Image info: ', img_id, img_name, img_width, img_height, px_x, px_y)
            print(img_arr.shape, img_arr.dtype, np.min(img_arr), np.max(img_arr))

            img_save_path = images_dir / f"{img_name.split('.')[0]}.png"
            print(f"Image name is {img_name_noext}")
            img_to_save = Image.fromarray(img_arr)
            img_to_save.save(img_save_path)

        # now iterate through the rois and print their info and colours
        rois = getter.get_rois_for_image(img_id)
        for roi in rois:
            # roi_name = roi.getName()._val if hasattr(roi, "getName") else None
            # print(f'\nROI: {type(roi)}\n')
            # print(f'  ROI name: {roi_name}')
            for shape in roi.copyShapes():
                # shape_class = shape.getTextValue()._val if hasattr(shape, "getTextValue") else None
                # print(f'  Shape class: {shape_class}')
                shape_color = shape.getStrokeColor()._val if hasattr(shape, "getFillColor") else None
                print(shape_color)

                if shape_color == -65281:
                    detection_type = "granule"
                elif shape_color == 11927551:
                    detection_type = "cell"
                elif shape_color == -1744764929:
                    detection_type = granule_type

                print(f'  Shape color: {shape_color}')
                shape_x = shape.getX()._val if hasattr(shape, "getX") else None
                shape_y = shape.getY()._val if hasattr(shape, "getY") else None
                shape_width = shape.getWidth()._val if hasattr(shape, "getWidth") else None
                shape_height = shape.getHeight()._val if hasattr(shape, "getHeight") else None
                print(f'  Shape bbox: x={shape_x:.2f}, y={shape_y:.2f}, width={shape_width:.2f}, height={shape_height:.2f}\n')

                bbox_x = float(shape_x)
                bbox_y = float(shape_y)
                bbox_w = float(shape_width)
                bbox_h = float(shape_height)

                # csv_file.write(f"{dataset_ids[i]},")
                csv_file.write(f"0000,")  # writing dataset id as 0000 cuz i cba, correct them while annotating in qupath
                csv_file.write(f"{img_name},")
                csv_file.write(f"{img_id},")
                csv_file.write(f"{px_x.getValue()},")
                csv_file.write(f"{px_unit},")
                csv_file.write(f"{detection_type},")
                csv_file.write(f"{shape_color},")
                csv_file.write(f"{bbox_x},")
                csv_file.write(f"{bbox_y},")
                csv_file.write(f"{bbox_w},")
                csv_file.write(f"{bbox_h}\n")

    csv_file.close()

3014263: 20261027 14:03:32 -INFO-: Logger already setup and ready to go!


2026-10-27 14:03:32,455 INFO  [                              omero_test] (MainThread) Logger already setup and ready to go!


3014263: 20261027 14:03:32 -INFO-: Opening connection to OMERO with token: 30875857-f342-4083-ad20-71578168e6df, hostname: omero-cci-cli.gu.se


2026-10-27 14:03:32,458 INFO  [                              omero_test] (MainThread) Opening connection to OMERO with token: 30875857-f342-4083-ad20-71578168e6df, hostname: omero-cci-cli.gu.se
2026-10-27 14:03:32,461 DEBUG [                           omero.gateway] (MainThread) omero-cci-cli.gu.se
2026-10-27 14:03:32,462 DEBUG [                           omero.gateway] (MainThread) 4064
2026-10-27 14:03:32,464 DEBUG [                           omero.gateway] (MainThread) []
2026-10-27 14:03:32,470 DEBUG [                           omero.gateway] (MainThread) Connect attempt, sUuid=30875857-f342-4083-ad20-71578168e6df, group=None, self.sUuid=None
2026-10-27 14:03:32,472 DEBUG [                           omero.gateway] (MainThread) connected? False


2026-10-27 14:03:32,496 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.client.uuid' to 'a7e86534-31c9-43bb-87d4-d64616a9449a'
2026-10-27 14:03:32,497 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.session.uuid' to '30875857-f342-4083-ad20-71578168e6df'
2026-10-27 14:03:32,498 DEBUG [                     omero.gateway.utils] (MainThread) Key 'omero.group' not found in <ServiceOptsDict: {'omero.client.uuid': 'a7e86534-31c9-43bb-87d4-d64616a9449a', 'omero.session.uuid': '30875857-f342-4083-ad20-71578168e6df'}>
2026-10-27 14:03:32,498 DEBUG [                     omero.gateway.utils] (MainThread) Key 'omero.user' not found in <ServiceOptsDict: {'omero.client.uuid': 'a7e86534-31c9-43bb-87d4-d64616a9449a', 'omero.session.uuid': '30875857-f342-4083-ad20-71578168e6df'}>
2026-10-27 14:03:32,498 DEBUG [                           omero.gateway] (MainThread) Joined Session OK with Uuid: 30875857-f342-4083-ad20-71578168e6df
2026-10-27 14:03

3014263: 20261027 14:03:32 -INFO-: Closing connection to OMERO with token: 30875857-f342-4083-ad20-71578168e6df


... (75 outputs left out)

2026-10-27 14:04:40,614 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.client.uuid' to 'a7e86534-31c9-43bb-87d4-d64616a9449a'
2026-10-27 14:04:40,615 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.session.uuid' to '30875857-f342-4083-ad20-71578168e6df'
2026-10-27 14:04:40,616 DEBUG [                     omero.gateway.utils] (MainThread) Setting 'omero.group' to 553
2026-10-27 14:04:40,651 INFO  [                           omero.gateway] (MainThread) Registered 30875857-f342-4083-ad20-71578168e6df/e1455b23-e5f8-4d3c-a261-18af0261bd3aomero.api.RawPixelsStore -t -e 1.1:tcp -h 10.129.6.110 -p 45447 -t 60000


11927551
  Shape color: 11927551
  Shape bbox: x=421.70, y=797.88, width=1614.74, height=1386.35

11927551
  Shape color: 11927551
  Shape bbox: x=1872.11, y=2324.24, width=1597.26, height=1553.85

-65281
  Shape color: -65281
  Shape bbox: x=1702.46, y=1561.74, width=61.74, height=58.27

-1744764929
  Shape color: -1744764929
  Shape bbox: x=1088.45, y=1929.95, width=64.35, height=64.35




2026-10-27 14:04:42,577 INFO  [                           omero.gateway] (MainThread) Unregistered 30875857-f342-4083-ad20-71578168e6df/e1455b23-e5f8-4d3c-a261-18af0261bd3aomero.api.RawPixelsStore -t -e 1.1:tcp -h 10.129.6.110 -p 45447 -t 60000


Image info:  18864 40135-P2_x4300_-10_0064.ome.tiff 4096 4096 3.2240475390288066 NANOMETER 3.2240475390288066 NANOMETER
(4096, 4096) uint16 1008 2789
Image name is 40135-P2_x4300_-10_0064
11927551
  Shape color: 11927551
  Shape bbox: x=571.77, y=1034.13, width=2162.74, height=1776.72

-1744764929
  Shape color: -1744764929
  Shape bbox: x=771.45, y=2039.62, width=40.70, height=37.00



In [23]:
# verify that all the files in the csv file exist in the images_dir

import pandas as pd
import os

for file in os.listdir(images_dir):
    if not file.endswith(".csv"):
        continue
    print(file)
    df = pd.read_csv(images_dir / file)

rare_granules_raw.csv
rare_granules_raw_correct.csv
rare_granules_raw_correct_beforenewspikes.csv


In [9]:
df

,dataset_id,image name,image id,pixel size,pixel unit,class name,color,x topleft pixels,y topleft pixels,width pixels,height pixels
0,2251,40135_E4_new_x4300_-10um_0001.ome.tif,20133,3.224048,NANOMETER,granule,-1744764929,2757.000000,2185.500000,67.500000,84.000000
1,2251,40135_E4_new_x4300_-10um_0001.ome.tif,20133,3.224048,NANOMETER,granule,-1744764929,2082.000000,1830.000000,64.500000,82.500000
2,2251,40135_E4_new_x4300_-10um_0001.ome.tif,20133,3.224048,NANOMETER,granule,-1744764929,1638.000000,1687.500000,58.500000,64.500000
3,2251,40135_E4_new_x4300_-10um_0001.ome.tif,20133,3.224048,NANOMETER,granule,-1744764929,1596.000000,2430.000000,69.000000,84.000000
4,2251,40135_E4_new_x4300_-10um_0001.ome.tif,20133,3.224048,NANOMETER,granule,-1744764929,1686.000000,2589.000000,51.000000,55.500000
...,...,...,...,...,...,...,...,...,...,...,...
137,1226,40135_G1_x4300_-10um_0024.ome.tiff,15653,3.224048,NANOMETER,granule,-1744764929,2104.172007,2793.278152,53.604974,57.131644
138,1226,40135_G1_x4300_-10um_0024.ome.tiff,15653,3.224048,NANOMETER,granule,-1744764929,1509.990307,1633.321057,1142.432364,1402.076173
139,1226,40135_G1_x4300_-10um_0024.ome.tiff,15653,3.224048,NANOMETER,spike,-1744764929,1937.600000,2231.250000,67.375000,56.350000
140,1226,40135_G1_x4300_-10um_0002.ome.tiff,15686,3.224048,NANOMETER,spike,-1744764929,2690.636123,1803.488251,53.077224,75.824605


In [10]:
all_names = np.unique(df['image name'].tolist())
all_names = [f.split('.')[0] for f in all_names]
all_names

['40135-B5_x4300_-10um_0025',
 '40135_E1_x4300_-10um_0013',
 '40135_E4_new_x4300_-10um_0001',
 '40135_E4_new_x4300_-10um_0021',
 '40135_E4_new_x4300_-10um_0023',
 '40135_E4_new_x4300_-10um_0027',
 '40135_E4_new_x4300_-10um_0029',
 '40135_E4_new_x4300_-10um_0044',
 '40135_E4_new_x4300_-10um_0053',
 '40135_E4_new_x4300_-10um_0064',
 '40135_E4_new_x4300_-10um_0082',
 '40135_E4_new_x4300_-10um_0097',
 '40135_G1_x4300_-10um_0002',
 '40135_G1_x4300_-10um_0018',
 '40135_G1_x4300_-10um_0024',
 '40135_G1_x4300_-10um_0027',
 '40135_G1_x4300_-10um_0033',
 '40135_G1_x4300_-10um_0091',
 '40135_G1_x4300_-10um_0093',
 '40135_G1_x4300_-10um_0095',
 '40135_G1_x4300_-10um_0101',
 '40135_J5_new_x4300_-10um_0002',
 '40135_J5_new_x4300_-10um_0081']

In [11]:
filenames = [f.split('.')[0]for f in os.listdir(images_dir) if f.endswith(".png")]
filenames

['40135_E1_x4300_-10um_0013',
 '40135_J5_new_x4300_-10um_0081',
 '40135_G1_x4300_-10um_0002',
 '40135_G1_x4300_-10um_0101',
 '40135_E4_new_x4300_-10um_0053',
 '40135_G1_x4300_-10um_0095',
 '40135_E4_new_x4300_-10um_0023',
 '40135_G1_x4300_-10um_0093',
 '40135_E4_new_x4300_-10um_0044',
 '40135_E4_new_x4300_-10um_0064',
 '40135_E4_new_x4300_-10um_0021',
 '40135_G1_x4300_-10um_0033',
 '40135-B5_x4300_-10um_0025',
 '40135_G1_x4300_-10um_0024',
 '40135_E4_new_x4300_-10um_0029',
 '40135_E4_new_x4300_-10um_0001',
 '40135_G1_x4300_-10um_0091',
 '40135_E4_new_x4300_-10um_0097',
 '40135_E4_new_x4300_-10um_0027',
 '40135_G1_x4300_-10um_0027',
 '40135_J5_new_x4300_-10um_0002',
 '40135_E4_new_x4300_-10um_0082',
 '40135_G1_x4300_-10um_0018']

In [12]:
len(all_names), len(filenames)

(23, 23)

In [26]:
filenames[0].find("_")

4

In [14]:
len(np.unique(image_ids_tails + image_ids_spikes))

23

In [15]:
missing_images = []
for n in all_names:
    if n not in filenames:
        missing_images.append(n)

if missing_images:
    print(f"Missing images: {missing_images}")

In [16]:
missing_images = []
for n in filenames:
    if n not in all_names:
        missing_images.append(n)

if missing_images:
    print(f"Missing images: {missing_images}")

In [17]:
missing_images

[]